# How Decoding Depends on the Error Probability

Here we vary the error probability `p` while keeping the code length and iteration limit fixed. For each value, we generate independent frames and process them with Rust `decode-batch`. We measure the exact-recovery rate, the fraction of frames with a zero syndrome, and the mean number of iterations.

This demonstration uses RA-6000. Increase `FRAMES` for a more reliable estimate, or reduce it for faster experiments.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261113
FRAMES = 16
P_VALUES = [0.002, 0.005, 0.01, 0.02]
THREADS = 4
MAX_ITER = 100
rng = np.random.default_rng(SEED)

# Fix one valid codeword; choose it once and transmit it in every frame.
k = 120
information = "".join(rng.integers(0, 2, size=k).astype(str))
codeword = encode(MATRIX_RA, information)
print(f"n={len(codeword)}, k={k}, rate={k/len(codeword):.4f}; reusing the same codeword for the sweep")

## Channel Simulation and Metrics

Let `E` be an independent error mask with `P(Eⱼ=1)=p`, and let `B` be Bob's random bit string. We form `A=B XOR E` and the public message `M=C XOR B`. Then `A XOR M=C XOR E`, exactly matching the binary symmetric channel model. For each result, we count **exact recovery** (`estimate == C`) and `success` (the syndrome passed). These outcomes are not equivalent.


In [ ]:
rows = []
for p in P_VALUES:
    errors = rng.random((FRAMES, len(codeword))) < p
    bobs = rng.integers(0, 2, size=errors.shape, dtype=np.uint8)
    c = np.fromiter((int(bit) for bit in codeword), dtype=np.uint8)
    alice = bobs ^ errors.astype(np.uint8)
    public = bobs ^ c
    a_frames = ["".join(row.astype(str)) for row in alice]
    m_frames = ["".join(row.astype(str)) for row in public]
    results = decode_batch(MATRIX_RA, a_frames, m_frames, p,
                           max_iter=MAX_ITER, threads=THREADS)
    exact = [r.estimate == codeword for r in results]
    rows.append({"p": p,
                 "exact_frame_rate": float(np.mean(exact)),
                 "syndrome_rate": float(np.mean([r.success for r in results])),
                 "mean_iterations": float(np.mean([r.iterations for r in results])),
                 "mean_ms_per_frame": float(np.mean([r.elapsed_ms for r in results]))})
    print(f"p={p:.3f}: exact={sum(exact)}/{FRAMES}, zero-syndrome={sum(r.success for r in results)}/{FRAMES}")

summary = pd.DataFrame(rows)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(summary.p, summary.exact_frame_rate, "o-", label="exact codeword")
axes[0].plot(summary.p, summary.syndrome_rate, "s--", label="zero syndrome")
axes[0].set(xlabel="Bit error probability p", ylabel="Fraction of frames", ylim=(-0.05, 1.05))
axes[0].grid(alpha=.3); axes[0].legend()
axes[1].plot(summary.p, summary.mean_iterations, "o-", color="darkorange")
axes[1].set(xlabel="Bit error probability p", ylabel="Mean iterations")
axes[1].grid(alpha=.3)
fig.suptitle("RA-6000: effect of channel quality")
fig.tight_layout()
plt.show()

## Interpretation

As `p` increases, decoding typically becomes harder: the iteration count and failure probability may rise. The exact threshold depends on the matrix and random sample. Results are noisy with few frames; increase `FRAMES` and change `SEED` to estimate a confidence interval.
